# 일기 → 프롬프트 변환 테스트

일기를 입력하면 ComfyUI용 픽셀아트 프롬프트로 변환합니다.

---

## 사용 방법
1. **셀 1** 실행 → Groq API 연결 확인
2. **셀 2** 실행 → 함수 및 프롬프트 설정 로드
3. **셀 3** 실행 → 일기 직접 입력 (한국어 OK) → 프롬프트 자동 생성
4. **셀 4** 실행 → 수정사항 입력 (선택, 한국어 OK) → 최종 프롬프트 출력
5. 출력된 **긍정 프롬프트**와 **부정 프롬프트**를 ComfyUI에 복붙

---

## 팀원 수정 가이드
- 프롬프트 스타일을 바꾸고 싶으면 **셀 2의 `FIXED_PREFIX` / `FIXED_SUFFIX`** 를 수정하세요
- 부정 프롬프트를 바꾸고 싶으면 **셀 2의 `NEGATIVE_PROMPT`** 를 수정하세요
- LLM에게 주는 지시를 바꾸고 싶으면 **셀 2의 `diary_to_prompt()` 함수 안 `content`** 를 수정하세요
- `.env` 파일에 `GROQ_API_KEY` 가 있어야 실행됩니다 (팀 공유 키 사용)

In [ ]:
# ============================================================
# 셀 1: Groq API 클라이언트 초기화
# ============================================================
# ai_engine 폴더 아래에 .env 파일을 만들고 API 키를 입력해야 합니다.
# .env 파일이 없거나 키가 없으면 에러가 납니다!
# ============================================================

import os
from groq import Groq
from dotenv import load_dotenv

load_dotenv()  # .env 파일에서 환경변수 로드
client = Groq(api_key=os.getenv('GROQ_API_KEY'))  # Groq 클라이언트 생성
print('Groq API 연결 완료')

In [ ]:
# ============================================================
# 셀 2: 프롬프트 설정 및 함수 정의
# ============================================================

# ▼ 픽셀아트 스타일 고정 토큰
FIXED_PREFIX = "(pixel art:1.2), (medium shot:1.4), (centered:1.2), low resolution, retro video game style, flat coloring, simplistic shapes, detailed with individual pixels"

# ▼ 시점 토큰 - 수정 가능 (예: isometric view:0.8, wide shot:1.2 등)
FIXED_SUFFIX = "(Close-up:0.8)"

# ▼ 부정 프롬프트
NEGATIVE_PROMPT = "(landscape focus:1.5), small figure, busy background, dark, neon, (realistic:1.3), 3d, distorted limbs"


def diary_to_prompt(diary_text):
    """
    한국어 일기 → ComfyUI용 픽셀아트 프롬프트 변환

    Args:
        diary_text (str): 사용자가 입력한 일기 (한국어 가능)

    Returns:
        str: FIXED_PREFIX + LLM 생성 장면 + FIXED_SUFFIX 형태의 완성된 프롬프트
    """
    response = client.chat.completions.create(
        # ▼ 사용할 LLM 모델 이름 - 바꾸면 다른 모델 사용 가능
        model='llama-3.3-70b-versatile',
        messages=[
            {
                "role": "user",
                "content": (
                    # ▼ LLM에게 주는 지시문 (영어로 작성해야 영어 프롬프트가 나옵니다)
                    # ▼ 아래 Rules 항목들을 수정하면 프롬프트 스타일이 달라집니다
                    f"Convert the following diary entry into a natural English image generation prompt for ComfyUI pixel art.\n" 
                    f"Rules:\n"
                    f"- The main character is always a young woman (the diary writer) and must be clearly visible and centered\n"  # 주인공: 항상 젊은 여성
                    f"- Write in natural descriptive phrases (not just keywords)\n"  # 자연스러운 문장 (키워드만 나열 금지)
                    f"- Up to 100 words, be vivid and expressive\n"  # 최대 100단어, 생동감 있게
                    f"- Describe the scene, mood, characters, setting, weather, and atmosphere in detail\n"  # 장면/분위기/날씨 등 묘사
                    f"- Output in English only, no extra explanation\n\n"  # 영어만 출력, 설명 없이
                    f"Diary: {diary_text}"
                )
            }
        ]
    )
    scene = response.choices[0].message.content.strip()  # LLM이 생성한 장면 설명 추출
    return f"{FIXED_PREFIX}, {scene}, {FIXED_SUFFIX}"  # 앞뒤 고정 토큰과 합치기


def modify_prompt(original_prompt, user_request):
    """
    기존 프롬프트에 사용자 요청 내용을 추가하여 보강
    기존 내용을 절대 삭제하거나 줄이지 않고, 요청한 요소만 추가/강조합니다.

    Args:
        original_prompt (str): 기존에 생성된 전체 프롬프트
        user_request (str): 추가/강조 요청 내용 (한국어 가능)

    Returns:
        str: 기존 내용을 유지하면서 요청 사항이 반영된 프롬프트
    """
    response = client.chat.completions.create(
        model='llama-3.3-70b-versatile',
        messages=[
            {
                "role": "user",
                "content": (
                    f"You are modifying an image generation prompt. Keep ALL the original description intact and only ADD or EMPHASIZE elements based on the request.\n"
                    f"Do NOT shorten, simplify, or remove any existing details.\n"
                    f"Original prompt: {original_prompt}\n"
                    f"Additional request (may be in Korean): {user_request}\n"  # 한국어 요청도 처리 가능
                    f"Rules: always keep the main character as a young woman, natural descriptive English, output the full modified prompt only."
                )
            }
        ]
    )
    return response.choices[0].message.content.strip()


def print_prompts(positive):
    """긍정/부정 프롬프트를 ComfyUI에 쓰기 편한 형태로 출력"""
    print(f"\n✅ 긍정 프롬프트:\n{positive}")
    print(f"\n❌ 부정 프롬프트:\n{NEGATIVE_PROMPT}")
    print("\n👆 위 프롬프트를 ComfyUI에 복붙해서 사용하세요!")


print('함수 준비 완료')

In [ ]:
# ============================================================
# 셀 3: 일기 입력 및 프롬프트 생성
# ============================================================
#   - 일기 내용을 입력하고 Enter → 여러 줄 입력 가능
#   - 입력이 끝나면 빈 줄에서 Enter 한 번 더 → 입력 완료
#   - 아무것도 입력하지 않고 바로 Enter → DEFAULT 예시로 실행
# ============================================================

# ▼ 아무것도 입력하지 않았을 때 사용되는 기본 예시 일기
DEFAULT = "오늘은 비가와서 집에서 독서를 했다.\n창밖에는 하루종일 비가 왔다."

print("일기 내용을 입력하세요 (Enter 두 번으로 완료, 바로 Enter 누르면 예시 사용):")
lines = []
while True:
    line = input()
    if line == "":  # 빈 줄 입력 시 종료
        break
    lines.append(line)

# 입력이 있으면 입력 내용 사용, 없으면 기본 예시 사용
diary_text = "\n".join(lines) if lines else DEFAULT
print(f"\n입력된 일기:\n{diary_text}\n")

# LLM을 호출해서 프롬프트 생성 (셀 2의 diary_to_prompt 함수 사용)
prompt = diary_to_prompt(diary_text)
print(f"생성된 프롬프트:\n{prompt}")

In [ ]:
# ============================================================
# 셀 4: 프롬프트 수정 (선택) 및 최종 출력
# ============================================================
# 생성된 프롬프트가 마음에 들지 않으면 수정 요청을 입력할 수 있습니다.
# 기존 내용은 유지되고 요청한 요소만 추가/강조됩니다.
# ============================================================

answer = input("수정사항이 있으신가요? (y/n): ").strip().lower()

if answer == 'y':
    user_request = input("내용을 입력해 주세요: ").strip()
    # 기존 프롬프트를 유지하면서 요청한 부분만 추가 (셀 2의 modify_prompt 함수 사용)
    prompt = modify_prompt(prompt, user_request)

# 긍정/부정 프롬프트 최종 출력
print_prompts(prompt)